# Prompt Ablation Study - 7. Mapping & Evaluation

This notebook runs mapping and evaluation for ALL 3 prompt strategies.

**Prerequisites:** Run notebooks 1-6 first (all source and BT QA).

## Environment Setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'sentence-transformers', 'nltk', 'pandas', 'matplotlib', 'seaborn'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from sentence_transformers import SentenceTransformer
print('Loading SBERT model...')
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
del sbert
print('SBERT model cached')

## Path Configuration

In [ ]:
import json

ABLATION_DIR = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/prompt-ablation'
BASELINE_EVAL = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/evaluation'
QG_PATH = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/QG/qwen-3b.jsonl'

STRATEGIES = ['P1-fewshot', 'P2-cot', 'P3-concise']
LANGUAGES = ['de', 'es', 'fr', 'ru', 'zh-CN']

print(f'ABLATION_DIR: {ABLATION_DIR}')
print(f'BASELINE_EVAL: {BASELINE_EVAL}')

## Verify QA Files Exist

In [ ]:
print('=== Checking QA files ===')
all_ok = True
for strategy in STRATEGIES:
    print(f'\n{strategy}:')
    # Source
    src_file = f'{ABLATION_DIR}/QA/{strategy}/source-{strategy}.jsonl'
    exists = os.path.exists(src_file)
    print(f"  {'✓' if exists else '✗'} source")
    if not exists: all_ok = False
    # BT
    for lang in LANGUAGES:
        bt_file = f'{ABLATION_DIR}/QA/{strategy}/bt-{lang}-{strategy}.jsonl'
        exists = os.path.exists(bt_file)
        print(f"  {'✓' if exists else '✗'} bt-{lang}")
        if not exists: all_ok = False
if all_ok:
    print('\n=== All QA files found! ===')
else:
    print('\n=== WARNING: Some files missing! ===')

## Mapping Function

In [ ]:
def create_mapping(strategy, qg_path, qa_source_path, qa_bt_paths, output_path):
    """Create mapping between QG, source QA, and BT QA."""
    # Load QG
    qg_data = []
    with open(qg_path, 'r', encoding='utf-8') as f:
        for line in f:
            qg_data.append(json.loads(line))
    
    # Load source QA
    src_lookup = {}
    with open(qa_source_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            src_lookup[data['src']] = data['answers']
    
    # Load BT QA per language
    bt_lookups = {}
    for lang, path in qa_bt_paths.items():
        bt_lookups[lang] = {}
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    key = (data['src'], data['bt_tgt'])
                    bt_lookups[lang][key] = data['answers']
    
    # Create mapping
    mapped = []
    for item in qg_data:
        src = item['src']
        lang = item.get('lang_tgt', '')
        bt_tgt = item.get('bt_tgt', '')
        
        entry = {
            'src': src,
            'tgt': item.get('tgt', ''),
            'lang_tgt': lang,
            'bt_tgt': bt_tgt,
            'severity': item.get('severity', 'Unknown'),
            'questions': item.get('questions', ''),
            'answers_src': src_lookup.get(src, []),
            'answers_bt': bt_lookups.get(lang, {}).get((src, bt_tgt), []),
            'strategy': strategy
        }
        mapped.append(entry)
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in mapped:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    
    print(f'Mapping saved: {output_path} ({len(mapped)} entries)')
    return len(mapped)

## Run Mapping for All Strategies

In [ ]:
for strategy in STRATEGIES:
    print(f'\n=== Mapping {strategy} ===')
    
    # Check for CLEAN source first
    clean_src = f'{ABLATION_DIR}/QA/{strategy}/clean/clean-source-{strategy}.jsonl'
    if os.path.exists(clean_src):
        qa_source = clean_src
        print(f"  [INFO] Using CLEAN source file: {qa_source}")
    else:
        qa_source = f'{ABLATION_DIR}/QA/{strategy}/source-{strategy}.jsonl'
        print(f"  [INFO] Using STANDARD source file: {qa_source}")

    # Check for CLEAN BT
    qa_bt_paths = {}
    for lang in LANGUAGES:
        clean_bt = f'{ABLATION_DIR}/QA/{strategy}/clean/clean-bt-{lang}-{strategy}.jsonl'
        if os.path.exists(clean_bt):
            qa_bt_paths[lang] = clean_bt
            print(f"  [INFO] Using CLEAN BT file for {lang}")
        else:
            qa_bt_paths[lang] = f'{ABLATION_DIR}/QA/{strategy}/bt-{lang}-{strategy}.jsonl'
            print(f"  [INFO] Using STANDARD BT file for {lang}")

    mapping_output = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
    
    os.makedirs(f'{ABLATION_DIR}/{strategy}', exist_ok=True)
    create_mapping(strategy, QG_PATH, qa_source, qa_bt_paths, mapping_output)

## Copy Evaluation Scripts

In [ ]:
import shutil

# Get evaluation scripts from baseline
baseline_eval_scripts = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/evaluation'

for strategy in STRATEGIES:
    eval_dir = f'{ABLATION_DIR}/{strategy}/evaluation'
    os.makedirs(eval_dir, exist_ok=True)
    
    for script in ['sbert.py', 'string_comparison.py']:
        src = f'{baseline_eval_scripts}/{script}'
        if os.path.exists(src):
            shutil.copy(src, eval_dir)
            print(f'Copied {script} to {strategy}/evaluation/')

## Run SBERT Evaluation

In [ ]:
for strategy in STRATEGIES:
    print(f'\n=== SBERT {strategy} ===')
    mapping_file = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
    output_dir = f'{ABLATION_DIR}/{strategy}/evaluation/sbert'
    
    # Use the custom script
    sbert_script = f'{ABLATION_DIR}/evaluation/sbert.py'
    
    cmd = [
        sys.executable, '-u',
        sbert_script,
        '--input_path', mapping_file,
        '--output_dir', output_dir
    ]
    subprocess.run(cmd, check=True)
    print(f'Done {strategy}!')

## Run String Comparison Evaluation

In [ ]:
for strategy in STRATEGIES:
    print(f'\n=== String Comparison {strategy} ===')
    mapping_file = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
    output_dir = f'{ABLATION_DIR}/{strategy}/evaluation/string-comparison'
    
    # Use the custom script
    sc_script = f'{ABLATION_DIR}/evaluation/string_comparison.py'
    
    cmd = [
        sys.executable, '-u',
        sc_script,
        '--input_path', mapping_file,
        '--output_dir', output_dir
    ]
    subprocess.run(cmd, check=True)
    print(f'Done {strategy}!')

## Results Comparison

In [ ]:
import pandas as pd

print('\n' + '='*60)
print('ABLATION STUDY RESULTS')
print('='*60)

results = []

# Load baseline
baseline_sbert = f'{BASELINE_EVAL}/sbert_summary_by_lang.csv'
baseline_sc = f'{BASELINE_EVAL}/string_comparison_summary_by_lang.csv'

if os.path.exists(baseline_sbert):
    sbert_df = pd.read_csv(baseline_sbert)
    avg_sbert = sbert_df['avg_similarity'].mean()
else:
    avg_sbert = None

if os.path.exists(baseline_sc):
    sc_df = pd.read_csv(baseline_sc)
    avg_f1 = sc_df['avg_f1'].mean()
    avg_em = sc_df['avg_em'].mean()
else:
    avg_f1, avg_em = None, None

results.append({'strategy': 'baseline', 'sbert': avg_sbert, 'f1': avg_f1, 'em': avg_em})

# Load ablation results
for strategy in STRATEGIES:
    sbert_file = f'{ABLATION_DIR}/{strategy}/evaluation/sbert_summary_by_lang.csv'
    sc_file = f'{ABLATION_DIR}/{strategy}/evaluation/string_comparison_summary_by_lang.csv'
    
    if os.path.exists(sbert_file):
        sbert_df = pd.read_csv(sbert_file)
        avg_sbert = sbert_df['avg_similarity'].mean()
    else:
        avg_sbert = None
    
    if os.path.exists(sc_file):
        sc_df = pd.read_csv(sc_file)
        avg_f1 = sc_df['avg_f1'].mean()
        avg_em = sc_df['avg_em'].mean()
    else:
        avg_f1, avg_em = None, None
    
    results.append({'strategy': strategy, 'sbert': avg_sbert, 'f1': avg_f1, 'em': avg_em})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Git Push

In [ ]:
os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'config', '--global', 'user.email', 'simone@example.com'])
subprocess.run(['git', 'config', '--global', 'user.name', 'Simone'])
subprocess.run(['git', 'add', '-A'])
subprocess.run(['git', 'commit', '-m', 'Add prompt ablation mapping and evaluation results'])
subprocess.run(['git', 'push', 'origin', 'main'])
print('Push complete!')

## Summary

In [ ]:
print('\n' + '='*60)
print('ABLATION STUDY COMPLETE')
print('='*60)
print(f'\nResults saved to: {ABLATION_DIR}')
print('\nFor each strategy (P1-fewshot, P2-cot, P3-concise):')
print('  - mapping.jsonl')
print('  - evaluation/sbert/')
print('  - evaluation/string-comparison/')